# KLA Image Restoration - Google Colab Training Pipeline

This notebook provides two methods to set up your code and dataset for training on Google Colab:

### Option A: Local Browser Upload (No Google Drive Access Required)
1. Click the **Folder icon (📁)** on the left sidebar to open the Files panel.
2. Drag and drop `i4c_project.zip` and `train_dataset.zip` directly into that panel.

### Option B: Mount Google Drive (Fetch from Drive)
1. Upload `i4c_project.zip` and `train_dataset.zip` to the root folder of your Google Drive (`My Drive`).
2. Run the optional **Mount Google Drive** cell below to copy them to Colab automatically.

### Optional: Mount Google Drive & Copy Files (Run ONLY if using Option B)

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy files from Drive root to Colab session root
# (Adjust folder path if you put them inside a specific Drive folder)
import os
if os.path.exists('/content/drive/MyDrive/i4c_project.zip'):
    !cp /content/drive/MyDrive/i4c_project.zip /content/i4c_project.zip
if os.path.exists('/content/drive/MyDrive/train_dataset.zip'):
    !cp /content/drive/MyDrive/train_dataset.zip /content/train_dataset.zip
print("Copy complete!")

## 1. Unzip Code and Dataset
Extract the archives directly into Colab's fast local SSD storage.

In [ ]:
import os

project_zip = 'i4c_project.zip'
dataset_zip = 'train_dataset.zip'

print("Checking for uploaded ZIP files...")
if not os.path.exists(project_zip):
    raise FileNotFoundError(f"i4c_project.zip not found! Please check your upload/copy method.")
if not os.path.exists(dataset_zip):
    raise FileNotFoundError(f"train_dataset.zip not found! Please check your upload/copy method.")

print("Extracting project repository...")
!unzip -q {project_zip} -d i4c

print("Extracting dataset...")
!unzip -q {dataset_zip} -d dataset
print("Extraction complete.")

## 2. Configure Scaffolding & Directory Setup
Create the links inside `i4c/data/raw/` pointing to the training dataset so that the code resolves paths correctly.

In [ ]:
import os

# Create raw data directories inside i4c
os.makedirs('i4c/data/raw/train', exist_ok=True)

# Create links from the extracted dataset to i4c data paths
!ln -sfn /content/dataset/train/GT /content/i4c/data/raw/train/GT
!ln -sfn /content/dataset/train/NoisyLR /content/i4c/data/raw/train/NoisyLR

print("Verification of linked directories:")
print("GT exists:", os.path.exists('i4c/data/raw/train/GT/000000.npy'))
print("NoisyLR exists:", os.path.exists('i4c/data/raw/train/NoisyLR/000000.npy'))

## 3. Install Dependencies
Install PyTorch metrics and logging libraries on the Colab instance.

In [ ]:
!pip install -q lpips torchmetrics pyyaml tqdm opencv-python matplotlib

## 4. Execute Training Loop
Run the PyTorch training loop on the Colab GPU. Mixed Precision (AMP) is enabled automatically.

In [ ]:
%cd /content/i4c
%env PYTHONPATH=.

# Run the training script
!python train.py

## 5. Download Model Checkpoint and Training Log
Run this cell at the end of training to download the best checkpoint `model_best.pt` and the training log CSV directly to your local computer's Downloads folder through your browser.

In [ ]:
from google.colab import files
import os

best_model_path = '/content/i4c/weights/model_best.pt'
log_path = '/content/i4c/outputs/training_log.csv'

print("Downloading best model weights...")
if os.path.exists(best_model_path):
    files.download(best_model_path)
else:
    print("Best model weights file not found!")

print("Downloading training metrics log...")
if os.path.exists(log_path):
    files.download(log_path)
else:
    print("Training log file not found!")